# ITS AI Codellama-7B-Instruct QLoRA finetuning

NOTE: Before we begin, ensure you already have access to T4 GPU in Kaggle. You can do so by verification using phone number or persona verification. If not, you won't be able to train the codellama using a GPU.

[1] First we need ensure the current environment session is running on T4 GPU and we need to install the packages first

```!nvidia-smi```

```!pip install -q transformers accelerate peft trl bitsandbytes datasets```

[2] Next, import the datasets on the right sidebar. Currently we have to import soal_ujian.json and nilai_ujian.json, the datasets that has been processed by Harmoni (shoutout to harmoni).
If you already have processed data (e.g. train_data.jsonl), upload that as well

[3] Login to HuggingFace via Kaggle Secrets
Add-ons -> Secrets -> add a secret named HF_TOKEN with your own Hugging Face token. This is required because codellama-7b-instruct model is gated by HF


In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

[4] Next we are going to check the datasets folder pathname first

In [2]:
# CHECK THE ACTUAL FOLDER/FILE PATHNAME FIRST

import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/hanzfr/soal-ujian/soal_ujian.json
/kaggle/input/datasets/hanzfr/train-data/train_data.jsonl
/kaggle/input/datasets/hanzfr/nilai-ujian/nilai_ujian.json


[5] then we're going to rebuild the datasets. the purpose of this is to adjust with the finetuning data format

In [9]:
import json

# change the path according to your preferred set filepath
with open("/kaggle/input/datasets/hanzfr/soal-ujian/soal_ujian.json") as f:
    soal = json.load(f)
with open("/kaggle/input/datasets/hanzfr/nilai-ujian/nilai_ujian.json") as f:
    nilai = json.load(f)

soal_by_id = {s["id"]: s for s in soal}

def normalize_scores(nilai_dict):
    if max(nilai_dict.values()) <= 10:
        return {k: v * 10 for k, v in nilai_dict.items()}
    return nilai_dict

dataset = []
for n in nilai:
    q = soal_by_id.get(n["id_soal"])
    if q is None:
        continue
    scores = normalize_scores(n["nilai"])
    avg = round(sum(scores.values()) / len(scores), 2)
    dataset.append({
        "id_soal": n["id_soal"],
        "soal": q["soal"],
        "expected_output": q["expected_output"],
        "kode_siswa": n["kode_siswa"],
        "level_siswa": n["level_siswa"],
        "nilai": scores,
        "nilai_avg": avg,
        "feedback": n["feedback"],
    })

print(f"Usable examples: {len(dataset)}")

def format_example(ex):
    prompt = (
        f"Soal: {ex['soal']}\n"
        f"Output yang diharapkan: {ex['expected_output']}\n\n"
        f"Kode siswa:\n```python\n{ex['kode_siswa']}\n```\n\n"
        f"Nilai kode siswa ini dan berikan feedback."
    )
    response = (
        f"Penilaian:\n"
        + "\n".join(f"- {k}: {v}" for k, v in ex["nilai"].items())
        + f"\n\nRata-rata: {ex['nilai_avg']}\n\nFeedback: {ex['feedback']}"
    )
    return {"id_soal": ex["id_soal"], "text": f"<s>[INST] {prompt} [/INST] {response} </s>"}

formatted = [format_example(ex) for ex in dataset]

with open("/kaggle/working/train_data.jsonl", "w") as f:
    for row in formatted:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

Usable examples: 1512


[6] Check token lengths from the datasets. we will ensure there's no dataset that is exceeding the context window

In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("codellama/CodeLlama-7b-Instruct-hf")

lengths = []
with open("/kaggle/working/train_data.jsonl") as f:
    for line in f:
        row = json.loads(line)
        lengths.append(len(tokenizer(row["text"])["input_ids"]))

import statistics
print("min/max:", min(lengths), max(lengths))
print("mean:", statistics.mean(lengths))
print("p95:", sorted(lengths)[int(len(lengths)*0.95)])

min/max: 274 1151
mean: 510.9298941798942
p95: 842


[7] split the dataset into training and validation. we have to make sure codellama TRULY learns to generalize and not memorizing the training examples. preventing overfitting

In [14]:
import random

rows = [json.loads(l) for l in open("/kaggle/working/train_data.jsonl")]
ids = sorted(set(r["id_soal"] for r in rows))
random.seed(42)
random.shuffle(ids)
val_ids = set(ids[:int(len(ids)*0.15)])

train_rows = [r for r in rows if r["id_soal"] not in val_ids]
val_rows = [r for r in rows if r["id_soal"] in val_ids]
print(f"train: {len(train_rows)}, val: {len(val_rows)}")

with open("/kaggle/working/train_split.jsonl", "w") as f:
    for r in train_rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
with open("/kaggle/working/val_split.jsonl", "w") as f:
    for r in val_rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

train: 1296, val: 216


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

dataset = load_dataset("json", data_files={
    "train": "/kaggle/working/train_split.jsonl",
    "validation": "/kaggle/working/val_split.jsonl",
})

training_args = SFTConfig(
    output_dir="/kaggle/working/checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    max_seq_length=1024,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=lora_config,
)
trainer.train()
trainer.save_model("/kaggle/working/final_model")